# Drive 연결

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


# 문제 1

[문제 1] - 어느 건설 회사에서 콘크리트의 압축 강도에 영향을 미치는 요인이 무엇인지 확인하고자 한다.
1,030개 콘크리트 샘플을 대상으로 압축 강도와 관련된 8개 변수를 측정하였다.

[데이터]
- 파일명: 9_3_1.csv
- 출처: UCI Machine Learning Repository - Concrete Compressive Strength
  - 원본: I-Cheng Yeh (1998), https://archive.ics.uci.edu/dataset/165
  - 링크: https://doi.org/10.24432/C5PK67

[Data Description]
※ `9_3_1.csv`는 콘크리트의 압축 강도는 건축물의 안전성에 직접적인 영향을 미치는 핵심 지표이다.
이 데이터셋은 다양한 배합 조건에서 제조된 콘크리트의 압축 강도를 측정한 결과를
포함하며, 각 샘플은 고유한 ID로 식별된다.
콘크리트는 시멘트, 물, 골재 등의 혼합물로 구성되며, 각 성분의 비율과 양생 기간이
최종 압축 강도에 큰 영향을 미친다. 특히 물-시멘트 비율(W/C ratio)은 강도를
결정하는 가장 중요한 요소 중 하나로 알려져 있다.

[컬럼 설명]
- Cement: 시멘트 - 콘크리트의 주요 결합재
- Blast Furnace Slag: 고로 슬래그 - 시멘트의 일부를 대체하는 산업 부산물
- Fly Ash: 플라이 애쉬 - 화력발전소 부산물로 시멘트 대체재
- Water: 물 - 시멘트의 수화 반응을 위해 필요
- Superplasticizer: 고성능 감수제 - 작업성 향상 및 물 사용량 감소
- Coarse Aggregate: 굵은 골재 - 콘크리트의 뼈대 역할
- Fine Aggregate: 잔골재 - 콘크리트의 공극을 채움
- Age: 양생 기간 - 시간 경과에 따른 강도 증가
- Concrete compressive strength: 압축 강도 (MPa)

[문항]
(1) 콘크리트 압축 강도(MPa)를 종속 변수로 하여, 절편을 포함한 다중 선형회귀 모형을 적합하세요. 이때 분석에 불필요한 변수는 제외해야 하며, 회귀분석 결과 유의수준 5% 이하(p < 0.05)로 통계적으로 의미 있게 나타난 독립(설명)변수의 개수를 구하세요.

(2) 학습 데이터(샘플 번호 1번~721번)에 대해, 실제 콘크리트 압축 강도와 회귀모형의 예측값 사이의 피어슨 상관계수를 계산하세요. 단, 답은 소수점 셋째 자리에서 반올림해 주세요.
- 학습 세트: 총 721개 (샘플 ID 1~721)
- 평가(테스트) 세트: 309개 (샘플 ID 722~1030)

(3) 위에서 훈련된 회귀모델로 테스트 데이터(샘플 ID 722~1030)에 대한 예측을 수행한 후, 예측값과 실제값 간의 RMSE(평균제곱근오차)를 산출하세요. 단, 답은 소수점 셋째 자리에서 반올림해 주세요.





In [6]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.metrics import mean_squared_error

# 1) 데이터 로드
DATA_PATH = "/content/drive/MyDrive/Colab Notebooks/Udemy/빅데이터분석기사_파이썬/작업형_제3유형/9회/data/9_3_1.csv"
df = pd.read_csv(DATA_PATH)
# print(df.head())

# 문제 지침대로 데이터 분할
train = df.iloc[:721].copy()
test = df.iloc[721:].copy()

# 독립변수, 종속변수 설정
X_cols = ['Cement', 'Blast Furnace Slag', 'Fly Ash', 'Water', 'Superplasticizer', 'Coarse Aggregate', 'Fine Aggregate', 'Age']

y_col = "Concrete compressive strength"

X_train, y_train = train[X_cols], train[y_col]
X_test, y_test = test[X_cols], test[y_col]

# (1) 다중선형회귀, 유의수준 5%에서 유의한 변수 개수
X_train_const = sm.add_constant(X_train)  # 절편 추가
model = sm.OLS(y_train, X_train_const).fit() # OLS 적합

p_values = model.pvalues.drop("const") # 절편 p-value 제외
significant_vars_count = (p_values < 0.05).sum()
print(f"(1) 통계적으로 유의미한 독립변수 개수: {significant_vars_count}")

# (2) 학습 데이터에서 실제값 vs 예측값 피어슨 상관계수
y_train_pred = model.predict(X_train_const)
correlation_rounded = round(train[y_col].corr(y_train_pred), 2)
print(f"(2) 학습 데이터 실제값-예측값 상관계수: {correlation_rounded}")

# (3) 테스트 데이터 RMSE 계산 (소수점 둘째 자리 반올림)
X_test_const = sm.add_constant(X_test)
y_test_pred = model.predict(X_test_const)

mse = mean_squared_error(y_test, y_test_pred)
rmse = round(np.sqrt(mse), 2)
print(f"(3) 테스트 데이터 RMSE: {rmse}")

(1) 통계적으로 유의미한 독립변수 개수: 7
(2) 학습 데이터 실제값-예측값 상관계수: 0.78
(3) 테스트 데이터 RMSE: 8.56


# 문제 2
[문제 2] - 1912년 발생한 타이타닉호 침몰 사고는 역사상 가장 비극적인 해난 사고 중 하나로 기록되어 있습니다. 당시 사고에서 승객들의 생존 여부는 단순히 운에 의한 것이 아니라 성별, 나이, 사회적 지위(객실 등급) 등 다양한 요인에 의해 영향을 받았다는 가설이 존재합니다. 본 분석에서는 로지스틱 회귀 모델을 구축하여 승객의 다양한 인적 특성이 생존 가능성에 미치는 영향력을 통계적으로 검증하고, 특정 변수의 유의성과 영향력(오즈비)을 파악하고자 합니다. 이 과정에서 발생할 수 있는 데이터의 불완전성(결측치)을 해결하기 위해 엄격한 전처리 지침을 준수해야 합니다.

[데이터]
- 파일명: 9_3_2.csv
- 출처: UCI Machine Learning Repository - Titanic Dataset

[Data Description]
※ `9_3_2.csv`는 타이타닉호 승객들의 신상 정보와 생존 여부를 포함하고 있습니다. 데이터 분석 전 아래의 **[결측치 처리 지시사항]**을 반드시 준수하여 전처리를 수행해야 합니다.

[결측치 처리 지시사항]
반드시 아래 순서와 방법에 따라 결측치를 처리한 후 분석을 진행하시오.
1. 수치형 변수(Age, Fare 등): 해당 변수의 **중앙값(Median)**으로 결측치를 대체하시오.
2. 범주형 변수(Sex, Embarked 등): 해당 변수의 **최빈값(Mode) 중 두 번째로 빈도가 높은 값**으로 대체하시오.
   - 만약 빈도가 동일한 값이 여러 개일 경우 알파벳 순서가 앞선 값을 선택한다. (예: 'S'와 'C'의 빈도가 같고 둘 다 최빈값 다음으로 높다면 'C'를 선택)

[컬럼 설명]
- Survived: 생존 여부 (0: 사망, 1: 생존) - 종속변수
- Pclass: 객실 등급 (1, 2, 3)
- Sex: 성별 (male, female)
- Age: 나이 (세)
- Fare: 요금 (달러)
- Embarked: 승선 항구 (C, Q, S)

[문항]
(1) 전처리가 완료된 학습 데이터를 바탕으로 생존 여부(Survived)를 예측하는 로지스틱 회귀 분석을 수행하시오. 모델 구축 시 모든 독립변수(Pclass, Sex, Age, Fare, Embarked)를 포함하며, 분석 결과 중 '나이(Age)' 컬럼의 p-value를 구하시오. (결과는 소수점 다섯째 자리에서 반올림하여 넷째 자리까지 표기)

(2) 로지스틱 회귀 모델의 회귀 계수를 활용하여, 성별이 '여성(female)'인 승객 대비 '남성(male)'인 승객의 생존 확률 오즈비(Odds Ratio)를 구하시오. (결과는 소수점 다섯째 자리에서 반올림하여 넷째 자리까지 표기)

(3) 구축된 모델을 통해 각 승객이 생존할 것으로 예측되는 확률을 계산하고, 그 확률값이 0.25 이상인 승객은 총 몇 명인지 구하시오.




In [23]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

DATA_PATH = "/content/drive/MyDrive/Colab Notebooks/Udemy/빅데이터분석기사_파이썬/작업형_제3유형/9회/data/"

# 1. 데이터 로드
df = pd.read_csv(DATA_PATH + '9_3_2.csv')
# print(df.head())

# print(df.isna().sum())

# 결측치 처리
# median()
df['age'] = df['age'].fillna(df['age'].median())
# print(df.isna().sum())

# 범주형, 두번째 최빈값으로 대체
vc = df['embarked'].dropna().value_counts().rename_axis("val").reset_index(name="count").sort_values(["count"], ascending=False)
second_mode = vc.iloc[1]['val']
df['embarked'] = df['embarked'].fillna(second_mode)
# print(df.isna().sum())

# 로지스틱 회귀 적합
formula = "survived ~ C(pclass) + C(sex) + age + fare + C(embarked)"
model = smf.logit(formula=formula, data=df).fit()
# print(model.summary())

# (2) female 대시 male 오즈비
male_coef = model.params["C(sex)[T.male]"]
male_or = round(float(np.exp(male_coef)), 4)
# print(f"(2) 성별 male의 오즈비: {male_or}")

# (3) 생존확률 >= 0.25인 승객 수
y_prob = model.predict(df)
cnt = int((y_prob >= 0.25).sum())
print(f"(3) 생존 확률 0.25 이상인 승객 수: {cnt}")

Optimization terminated successfully.
         Current function value: 0.447830
         Iterations 6
(3) 생존 확률 0.25 이상인 승객 수: 468
